# J2 Après-Midi | > Prompt ... engineering

**Objectifs d'apprentissage :**
1. Découvrir et prendre en main l'API d'un Modèle de Langage (OpenAI).
2. Maîtriser les principes du *Prompt Engineering* appliqués aux sciences sociales (Zero-shot, Few-shot, Chain of Thought).
3. Comparer une méthode traditionnelle d'analyse de sentiment (VADER) à une méthode LLM.
4. Appliquer ces concepts pour annoter et classifier le jeu de données INA (collecté au module précédent).


## 1. Introduction à l'API OpenAI

Lorsqu'on utilise ChatGPT via le site web, l'interface graphique envoie en arrière-plan nos requêtes à l'API d'OpenAI. Nous allons automatiser cela directement en Python.

**Sécurité importante :** En temps normal, ne mettez **jamais** votre clé API directement dans le code public. Pour les besoins de cet atelier, vous allez copier-coller temporairement la clé fournie par les instructeurs directement dans la variable `api_key`.

In [ ]:
from openai import OpenAI
import pandas as pd

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except:
    print("Erreur d'initialisation. Vérifiez votre clé API.")

def analyze_text(text, system_prompt, model="gpt-5.4-nano", temperature=0.0):
    """
    Fonction simple pour envoyer un texte et un prompt système à l'API OpenAI.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            # temperature 0 = prévisible/déterministe, 1 = créatif/aléatoire
            temperature=temperature,  
            max_tokens=250
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur lors de l'appel à l'API : {e}"

## 2. Le Prompt Engineering 

Pour un politiste ou un sociologue, **le prompt est l'équivalent d'un protocole d'annotation (codebook)**. Si le prompt est ambigu, le codage de vos données sera mauvais.

Explorons trois techniques fondamentales sur un exemple court.

> 💡 **Note sur les modèles ouverts (Hugging Face) :** Bien que nous utilisions OpenAI ici pour des raisons de facilité, ces techniques s'appliquent aussi aux modèles locaux (Mistral, Llama). Attention cependant : un prompt qui fonctionne bien sur l'API commerciale (`gpt-5.4-nano`) devra souvent être ajusté pour un modèle ouvert de plus petite taille, car ils sont souvent plus sensibles à la formulation exacte !

### A. Zero-Shot Prompting
C'est la méthode la plus simple. On donne l'instruction sans fournir d'exemple préalable.

In [ ]:
text_to_analyze = "Le gouvernement doit intervenir massivement pour protéger nos services publics."

zero_shot_prompt = """
Tu es un expert en idéologie politique française.
Classe le texte suivant selon : GAUCHE, DROITE, ou CENTRE.
Réponds UNIQUEMENT par la catégorie.
"""
print("Zero-Shot :", analyze_text(text_to_analyze, zero_shot_prompt))

### B. Few-Shot Prompting & Hack-Time 🛠️
Fournir des exemples (*Few-Shot*) agit comme un **livre de codage (codebook)** pour l'IA. 

**À vous de jouer :** Complétez le prompt ci-dessous en ajoutant vos propres exemples pour aider le modèle à mieux comprendre la catégorie 'CENTRE' ou pour corriger ses biais.

In [ ]:
few_shot_prompt = """
Tu es un expert en idéologie politique française.
Classe les textes selon : GAUCHE, DROITE, CENTRE.

Exemples :
- Texte : "Il faut baisser les charges pour libérer l'économie."
  Catégorie : DROITE
- Texte : "L'urgence est d'augmenter le SMIC."
  Catégorie : GAUCHE
- Texte : "... (Votre exemple pour le centre ici) ..."
  Catégorie : CENTRE

À toi :
"""
print("Few-Shot :", analyze_text(text_to_analyze, few_shot_prompt))

### C. Chain of Thought (Chaîne de Pensée)
Demander au modèle de justifier son choix *avant* de donner la réponse finale améliore considérablement ses performances sur des tâches complexes. En sciences sociales, cela permet aussi de **tracer la décision (auditabilité)** et de mieux interpréter comment l'algorithme catégorise.

In [ ]:
cot_prompt = """
Analyse l'idéologie politique du texte (GAUCHE, DROITE, CENTRE).
Procède ainsi :
1. Raisonnement : Analyse brièvement les mots-clés du texte et associe-les aux grandes familles politiques.
2. Label : Termine ta réponse par 'LABEL: [Catégorie]'.
"""
print("Chain of Thought :\n", analyze_text(text_to_analyze, cot_prompt))

## 3. Cas pratique : Analyse de sentiment (Wikipedia)

L'analyse de sentiment classique s'appuie sur des dictionnaires (comme **VADER**). Ces dictionnaires comptent les mots positifs et négatifs dans une phrase. Regardons la différence entre l'approche classique (VADER) et l'approche par LLM sur une phrase complexe tirée d'une page Wikipédia.

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Une phrase avec une négation et une ironie (difficile pour les algos classiques)
wiki_phrase = "Le ministre n'a pas exactement brillé par son courage politique lors de la réforme."

# 1. Approche VADER (Méthode classique, très rapide et automatisable sur des millions de textes)
vader_score = sia.polarity_scores(wiki_phrase)
print("Score VADER :", vader_score)

# 2. Approche LLM (Méthode moderne, plus fine mais plus coûteuse)
llm_prompt = """
Analyse le sentiment de cette phrase politique (POSITIF, NEGATIF, NEUTRE).
Attention à l'ironie et aux négations.
"""
print("\nAnalyse LLM :\n", analyze_text(wiki_phrase, llm_prompt))

> 💡 **Méthodologie** : Dans la recherche actuelle, on utilise souvent **VADER** ou des modèles pré-entraînés locaux pour traiter rapidement de très grands corpus historiques (millions de lignes). Les **LLMs** (OpenAI) sont utilisés pour annoter de façon très fine des échantillons plus petits ou des concepts complexes (ex: repérer l'ironie, l'orientation idéologique).

## 4. Atelier de Codage : Le Projet INA / ZFE

À la fin de la séance précédente, nous avons évoqué la collecte des métadonnées de l'INA concernant les "Zones à Faibles Émissions" (ZFE) sur CNews et TF1.

Chargeons le corpus pré-collecté `ina_zfe.csv`.

In [ ]:
# Assurez-vous d'avoir téléchargé ou généré le fichier data/ina_zfe/ina_zfe.csv dans Colab
df_ina = pd.read_csv("data/ina_zfe/ina_zfe.csv")
display(df_ina.head())
print(f"Le jeu de données contient {len(df_ina)} émissions.")

Notre objectif de recherche : **Le titre de l'émission est-il favorable, défavorable, ou neutre vis-à-vis de la politique environnementale des ZFE ?**

In [ ]:
system_prompt_zfe = """
Tu es un assistant de recherche en sociologie des médias.
Ton but est d'annoter l'orientation d'un titre de journal télévisé vis-à-vis des politiques environnementales (ZFE).

Classe le titre selon 3 catégories :
- FAVORABLE : Le titre présente les ZFE comme une bonne chose ou une solution.
- DEFAVORABLE : Le titre souligne la colère, les problèmes, les interdictions ou le côté punitif.
- NEUTRE : Le titre est purement descriptif et factuel.

Réponds UNIQUEMENT par la catégorie (FAVORABLE, DEFAVORABLE, NEUTRE).
"""

### Hack-Time 🛠️

Le code ci-dessous parcourt les 5 premières lignes de notre DataFrame INA.
Complétez la boucle pour envoyer chaque titre de l'émission à l'API OpenAI avec notre `system_prompt_zfe`, puis stocker et afficher le résultat.

In [ ]:
resultats = []

for index, row in df_ina.head(5).iterrows():
    titre = row['titre']
    chaine = row['chaine']
    
    # 1. Appeler la fonction analyze_text avec system_prompt_zfe
    # 2. Stocker la réponse dans une variable `annotation`
    annotation = ""  # <-- Remplacez ceci par votre appel à la fonction
    
    # On affiche le résultat
    print(f"[{chaine}] {titre} \n-> {annotation}\n")
    resultats.append(annotation)



## Conclusion

Nous avons vu comment automatiser un protocole de recherche qualitatif sur un corpus quantitatif.

**Limites à retenir :**
1. **Biais du Modèle :** Le modèle d'OpenAI a été aligné par des humains (RLHF). Sa définition du "favorable" ou "défavorable" porte un biais.
2. **Reproductibilité :** Si l'API met à jour son modèle `gpt-5.4-nano`, vos résultats dans 6 mois pourraient différer.
3. **Validation Inter-Juges :** Ne faites jamais confiance au LLM à 100%. Il est crucial de faire coder 10% du corpus par un humain (vous !) et de calculer un score de fiabilité inter-juges avec l'IA.